# 🍇 Grape Dual-Model Training (Setup B)
Train **both** architectures on Grape diseases (4 classes including Healthy):
1. **Model 1**: `EfficientNet-B2` (Compound scaled lesion detection)
2. **Model 2**: `ResNet-50` (Deep residual feature extractor)

**Classes:**
1. Grape — Black Rot
2. Grape — Esca (Black Measles)
3. Grape — Leaf Blight
4. Grape — Healthy ✅

**Runtime:** GPU (T4 recommended)

## Step 1: Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('❌ Set Runtime -> Change runtime type -> T4 GPU')

## Step 2: Upload `splits_grape.zip`

In [ ]:
import zipfile, os, glob

os.makedirs('data', exist_ok=True)

# 1. Check for single zip or batch zips
batch_zips = sorted(glob.glob('grape_batch_*.zip'))

if os.path.exists('splits_grape.zip'):
    print('Extracting splits_grape.zip...')
    with zipfile.ZipFile('splits_grape.zip', 'r') as z:
        z.extractall('data')
elif batch_zips:
    print(f'Extracting {len(batch_zips)} batch zip files...')
    for bz in batch_zips:
        print(f'  Unzipping {bz}...')
        with zipfile.ZipFile(bz, 'r') as z:
            z.extractall('data')
else:
    print('❌ No dataset zip found! Upload either splits_grape.zip OR the grape_batch_*.zip files.')

# Verify extracted splits
for split in ['train', 'val', 'test']:
    sdir = f'data/splits_grape/{split}'
    if os.path.isdir(sdir):
        classes = os.listdir(sdir)
        total = sum(len(os.listdir(os.path.join(sdir, c))) for c in classes)
        print(f'✅ {split}: {total} images across {len(classes)} classes')
    else:
        print(f'⚠️ {split} directory not found yet')

## Step 3: Shared Setup & Weighted Sampler

In [ ]:
import time, copy
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm

DATA_DIR = 'data/splits_grape'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASS_NAMES = sorted(os.listdir(os.path.join(DATA_DIR, 'train')))
NUM_CLASSES = len(CLASS_NAMES)
DISPLAY_NAMES = [c.replace('Grape___', 'Grape — ').replace('_', ' ') for c in CLASS_NAMES]

def get_loaders(img_size, batch_size=32):
    train_tf = transforms.Compose([
        transforms.Resize((img_size + 24, img_size + 24)),
        transforms.RandomCrop(img_size),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.3),
        transforms.RandomRotation(25),
        transforms.ColorJitter(0.3, 0.3, 0.2, 0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    
    train_ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(DATA_DIR, 'val'),   eval_tf)
    test_ds  = datasets.ImageFolder(os.path.join(DATA_DIR, 'test'),  eval_tf)
    
    counts = [0] * NUM_CLASSES
    for _, lbl in train_ds.samples:
        counts[lbl] += 1
    cw = [1.0 / max(c, 1) for c in counts]
    sw = [cw[lbl] * (2.0 if os.path.basename(p).startswith('gvlid_') else 1.0) for p, lbl in train_ds.samples]
    sampler = WeightedRandomSampler(sw, len(sw), replacement=True)
    
    return (
        DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    )

def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, desc='Training', save_path=None):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_acc = 0.0
    best_wts = copy.deepcopy(model.state_dict())
    
    print(f'\n=== {desc} ({epochs} epochs) ===')
    for ep in range(epochs):
        t0 = time.time()
        model.train()
        tr_loss, tr_cor, tr_tot = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc=f'{desc} Epoch {ep+1}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item() * x.size(0)
            _, p = torch.max(out, 1)
            tr_tot += y.size(0)
            tr_cor += (p == y).sum().item()
            
        model.eval()
        va_loss, va_cor, va_tot = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                loss = criterion(out, y)
                va_loss += loss.item() * x.size(0)
                _, p = torch.max(out, 1)
                va_tot += y.size(0)
                va_cor += (p == y).sum().item()
        
        scheduler.step()
        val_acc = va_cor / va_tot
        star = ''
        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(model.state_dict())
            star = ' ⭐ BEST'
            if save_path:
                torch.save({'model_state_dict': best_wts}, save_path)
        print(f'Epoch {ep+1:>2}/{epochs} | Train {100*tr_cor/tr_tot:.1f}% | Val {100*val_acc:.1f}% | {time.time()-t0:.0f}s{star}')
        
    model.load_state_dict(best_wts)
    return model, best_acc

## Step 4: Train Model 1 — `EfficientNet-B2` 🍇

In [ ]:
tr_loader_eff, va_loader_eff, te_loader_eff = get_loaders(img_size=260, batch_size=32)

model_eff = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
for p in model_eff.features[:6].parameters():
    p.requires_grad = False
in_ftrs = model_eff.classifier[1].in_features
model_eff.classifier = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(in_ftrs, 512), nn.SiLU(),
    nn.Dropout(0.2), nn.Linear(512, NUM_CLASSES)
)
model_eff = model_eff.to(DEVICE)

# Train
model_eff, _ = train_model(model_eff, tr_loader_eff, va_loader_eff, epochs=12, lr=1e-3, desc='EfficientNet Phase 1', save_path='grape_efficientnet_b2.pth')
for p in model_eff.parameters(): p.requires_grad = True
model_eff, eff_acc = train_model(model_eff, tr_loader_eff, va_loader_eff, epochs=8, lr=1e-4, desc='EfficientNet Phase 2', save_path='grape_efficientnet_b2.pth')

# Save
torch.save({'model_state_dict': model_eff.state_dict()}, 'grape_efficientnet_b2.pth')
print(f'✅ Model 1 (EfficientNet-B2) saved! Best Val: {100*eff_acc:.2f}%')

## Step 5: Train Model 2 — `ResNet-50` 🍇

In [ ]:
tr_loader_res, va_loader_res, te_loader_res = get_loaders(img_size=224, batch_size=32)

model_res = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for name, param in model_res.named_parameters():
    if 'layer3' not in name and 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False
in_ftrs = model_res.fc.in_features
model_res.fc = nn.Sequential(
    nn.Dropout(0.4), nn.Linear(in_ftrs, 512), nn.ReLU(),
    nn.Dropout(0.2), nn.Linear(512, NUM_CLASSES)
)
model_res = model_res.to(DEVICE)

# Train
model_res, _ = train_model(model_res, tr_loader_res, va_loader_res, epochs=12, lr=1e-3, desc='ResNet-50 Phase 1', save_path='grape_resnet50.pth')
for p in model_res.parameters(): p.requires_grad = True
model_res, res_acc = train_model(model_res, tr_loader_res, va_loader_res, epochs=8, lr=1e-4, desc='ResNet-50 Phase 2', save_path='grape_resnet50.pth')

# Save
torch.save({'model_state_dict': model_res.state_dict()}, 'grape_resnet50.pth')
print(f'✅ Model 2 (ResNet-50) saved! Best Val: {100*res_acc:.2f}%')

## Step 6: Download Both Grape Models ⬇️

In [ ]:
from google.colab import files
print('Downloading grape_efficientnet_b2.pth...')
files.download('grape_efficientnet_b2.pth')

print('Downloading grape_resnet50.pth...')
files.download('grape_resnet50.pth')

print('\nPlace both files in: d:\\gt\\models\\')